# Agent: secuencia, paralelo y decisiones

El propietario cambia su flujo; conserva el mismo run y el mismo RunResult.

API **2.1.3**, kernel dedicado y ejecución offline. Los actores Python son operadores deterministas.

In [1]:
from pathlib import Path
import sys
tutorial_root = next(p / "tutorials" for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "tutorials/_offline.py").is_file())
sys.path.insert(0, str(tutorial_root))
from _offline import enforce_offline, blocked_attempts
enforce_offline()
import agentic_systems as toolkit
assert toolkit.__version__ == "2.1.3", "Instala el wheel local 2.1.3: no hay fallback a 2.1.2"


In [2]:
@toolkit.tool
def increment(value: int) -> dict:
    """Suma uno de forma determinista."""
    return {"value": value + 1}

@toolkit.tool
def double(value: int) -> dict:
    """Duplica el valor."""
    return {"value": value * 2}

arithmetic = toolkit.skill(name="arithmetic", tools=[increment, double])
increase = toolkit.Action("increase", "increment")
multiply = toolkit.Action("multiply", "double")

def choose_next(data):
    return "multiply" if data["value"] >= 0 else "END"

def combine(branches):
    return {"value": sum(row["value"] for row in branches.values())}

In [3]:
for backend in ("native", "langgraph"):
    sequential = toolkit.Agent("sequential", toolkit.Sequence((increase, multiply)), skills=(arithmetic,), backend=backend)
    parallel = toolkit.Agent("parallel", toolkit.Parallel((increase, multiply), join=combine), skills=(arithmetic,), backend=backend)
    decision = toolkit.Agent("decision", toolkit.Graph(
        start="increase", nodes=(("increase", increase), ("multiply", multiply)),
        edges=(("increase", choose_next), ("multiply", "END")),
    ), skills=(arithmetic,), backend=backend)
    for agent, expected in ((sequential, 8), (parallel, 10), (decision, 8)):
        result = agent.run({"value": 3})
        assert result.ok and result.data == {"value": expected}
        print(backend, agent.name, {"input": 3, "output": result.data})
    assert decision.run({"value": -3}).data == {"value": -2}

native sequential {'input': 3, 'output': {'value': 8}}
native parallel {'input': 3, 'output': {'value': 10}}
native decision {'input': 3, 'output': {'value': 8}}


langgraph sequential {'input': 3, 'output': {'value': 8}}
langgraph parallel {'input': 3, 'output': {'value': 10}}
langgraph decision {'input': 3, 'output': {'value': 8}}


Secuencia: 3 → 4 → 8. Paralelo: el mismo 3 alimenta dos ramas → 4 y 6 → join suma 10. Decisión: 3 → 4 → multiply → 8; −3 → −2 → END. La condición recibe la salida del nodo anterior. El backend LangGraph construye un StateGraph real; no cambia el proveedor de agentes anidados.

In [4]:
assert blocked_attempts() == 0
print("PASS · inferencia: 0")

PASS · inferencia: 0
